In [0]:
DECLARE OR REPLACE VARIABLE budget_kes   DOUBLE DEFAULT 500000000;      -- KES 500M relief budget (example)
DECLARE OR REPLACE VARIABLE target_month DATE   DEFAULT DATE'2025-12-01'; -- season to allocate for

In [0]:
CREATE OR REPLACE TABLE workspace.climate_gold.resource_allocation
COMMENT 'Drought-relief budget allocation per eligible rural sub-county, weighted by risk x population'
AS
WITH eligible AS (
  SELECT r.as_of_month,
         d.county_name,
         d.district_name,
         r.district_id,
         r.risk_tier,
         r.risk_score,
         p.population,
         r.risk_score * p.population AS need_weight
  FROM workspace.climate_gold.district_risk_score r
  JOIN workspace.climate_silver.districts d            USING (district_id)
  JOIN workspace.climate_silver.district_population p  USING (district_id)
  WHERE r.as_of_month = target_month
    AND r.risk_tier IN ('High', 'Elevated')
    AND NOT p.is_urban
)
SELECT *,
       ROUND(100 * need_weight / SUM(need_weight) OVER (), 2)                     AS share_pct,
       ROUND(budget_kes * need_weight / SUM(need_weight) OVER (), 0)              AS allocated_kes,
       ROUND(budget_kes * need_weight / SUM(need_weight) OVER () / population, 0) AS kes_per_person,
       budget_kes                                                                 AS total_budget_kes
FROM eligible;

In [0]:
SELECT COUNT(*)                   AS eligible_sub_counties,
       SUM(population)            AS people_covered,
       ROUND(SUM(allocated_kes))  AS total_allocated_kes,
       MAX(total_budget_kes)      AS budget_kes
FROM workspace.climate_gold.resource_allocation;

In [0]:
SELECT county_name, district_name, risk_tier, risk_score, population,
       share_pct, allocated_kes, kes_per_person
FROM workspace.climate_gold.resource_allocation
ORDER BY allocated_kes DESC
LIMIT 15;

In [0]:
SELECT county_name,
       COUNT(*)              AS sub_counties,
       SUM(population)       AS people,
       SUM(allocated_kes)    AS allocated_kes,
       ROUND(SUM(share_pct), 1) AS share_pct
FROM workspace.climate_gold.resource_allocation
GROUP BY county_name
ORDER BY allocated_kes DESC;